In [89]:
from pathlib import Path
import shutil
import os
import subprocess
import json
from collections import defaultdict

In [ ]:

# 1) Change this to your project folder
root = Path(r"Path") # Path containing the subject data
# 2) Your subject folder / ID
sub = "217895"

ses = "V04"

type = "T1"

# This is where your subject DICOMs should live
dicom_dir = root / "Visit_2_all" / f"sub-{sub}" / f"ses-{ses}" / "DICOM" / type

# Helper output (for inspecting JSON sidecars)
helper_out = root / "Visit_2_all" / f"sub-{sub}"/ f"ses-{ses}" 

# Final BIDS output folder
bids_out = root / "Trial"/"bids"

# Config path (we will create / edit it)
config_path = root / "Visit_2_all" / f"sub-{sub}" / f"ses-{ses}" /"dcm2bids_config.json"

print("Project root:", root)
print("DICOM dir:", dicom_dir)
print("Helper out:", helper_out)
print("BIDS out:", bids_out)
print("Config path:", config_path)


Project root: C:\Users\abdul\Desktop\UCL_ANI\Dissertation\python-work
DICOM dir: C:\Users\abdul\Desktop\UCL_ANI\Dissertation\python-work\Visit_2_all\sub-217895\ses-V04\DICOM\T1
Helper out: C:\Users\abdul\Desktop\UCL_ANI\Dissertation\python-work\Visit_2_all\sub-217895\ses-V04
BIDS out: C:\Users\abdul\Desktop\UCL_ANI\Dissertation\python-work\Trial\bids
Config path: C:\Users\abdul\Desktop\UCL_ANI\Dissertation\python-work\Visit_2_all\sub-217895\ses-V04\dcm2bids_config.json


In [196]:
if not dicom_dir.exists():
    raise FileNotFoundError(f"DICOM folder not found: {dicom_dir}")


In [197]:
helper_out.mkdir(parents=True, exist_ok=True)

cmd = ["dcm2bids_helper", "-d", str(dicom_dir), "-o", str(helper_out)]
print("Running:", " ".join(cmd))

subprocess.run(cmd, check=True)


Running: dcm2bids_helper -d C:\Users\abdul\Desktop\UCL_ANI\Dissertation\python-work\Visit_2_all\sub-217895\ses-V04\DICOM\T1 -o C:\Users\abdul\Desktop\UCL_ANI\Dissertation\python-work\Visit_2_all\sub-217895\ses-V04


CompletedProcess(args=['dcm2bids_helper', '-d', 'C:\\Users\\abdul\\Desktop\\UCL_ANI\\Dissertation\\python-work\\Visit_2_all\\sub-217895\\ses-V04\\DICOM\\T1', '-o', 'C:\\Users\\abdul\\Desktop\\UCL_ANI\\Dissertation\\python-work\\Visit_2_all\\sub-217895\\ses-V04'], returncode=0)

In [198]:

help_2 = helper_out / "tmp_dcm2bids" / "helper"
print("Helper output created at:", help_2)
print("NIfTI files:", len(list(help_2.glob("*.nii*"))))
print("JSON files:", len(list(help_2.glob("*.json"))))



Helper output created at: C:\Users\abdul\Desktop\UCL_ANI\Dissertation\python-work\Visit_2_all\sub-217895\ses-V04\tmp_dcm2bids\helper
NIfTI files: 1
JSON files: 1


In [199]:
series_desc = set()
protocols = set()

json_files = sorted(help_2.glob("*.json"))
print("JSON sidecars found:", len(json_files))

for j in json_files:
    obj = json.loads(j.read_text(encoding="utf-8"))
    if "SeriesDescription" in obj:
        series_desc.add(obj["SeriesDescription"])
    if "ProtocolName" in obj:
        protocols.add(obj["ProtocolName"])

print("\nUnique SeriesDescription values:")
for s in sorted(series_desc):
    print("  -", s)

print("\nUnique ProtocolName values:")
for p in sorted(protocols):
    print("  -", p)

JSON sidecars found: 1

Unique SeriesDescription values:
  -  MPRAGE

Unique ProtocolName values:
  - Anon


In [200]:
T1_NAME = " MPRAGE"      # example
FLAIR_NAME = "SAG 3D T2 FLAIR (CUBE)"   # example

config = {
    "descriptions": [
        {
            "datatype": "anat",
            "suffix": "T1w",
            "criteria": {
                "SeriesDescription": T1_NAME
            }
        },
        {
            "datatype": "anat",
            "suffix": "FLAIR",
            "criteria": {
                "SeriesDescription": FLAIR_NAME
            }
        }
    ]
}

# 🔹 Where to save config

config_path.parent.mkdir(parents=True, exist_ok=True)

with open(config_path, "w") as f:
    json.dump(config, f, indent=4)

print("Config saved at:", config_path)

Config saved at: C:\Users\abdul\Desktop\UCL_ANI\Dissertation\python-work\Visit_2_all\sub-217895\ses-V04\dcm2bids_config.json


In [201]:

bids_out.mkdir(parents=True, exist_ok=True)

subprocess.run([
    "dcm2bids",
    "-d", dicom_dir,
    "-p", sub,
    "-s", ses,
    "-c", str(config_path),
    "-o", str(bids_out)
], check=True)

print("✅ BIDS created at:", bids_out)

✅ BIDS created at: C:\Users\abdul\Desktop\UCL_ANI\Dissertation\python-work\Trial\bids


In [57]:
"sub-" + sub +"/ses-BL"

'sub-75505/ses-BL'

In [ ]:
from pathlib import Path
import shutil
import os
import subprocess
import json
from collections import defaultdict

root = Path(r"C:\Users\abdul\Desktop\UCL ANI\Dissertation\python-work")

# 2) Your subject folder / ID
sub = "75505"

# This is where your subject DICOMs should live
dicom_dir = root / "sourcedata" / f"sub-{sub}" / "dicom"

# Helper output (for inspecting JSON sidecars)
helper_out = root / "sourcedata" / f"sub-{sub}" / "tmp_dcm2bids" / "helper"

# Final BIDS output folder
bids_out = root / "bids"

# Config path (we will create / edit it)
config_path = root / "dcm2bids_config.json"

bids_out.mkdir(parents=True, exist_ok=True)

cmd = [
    "dcm2bids_scaffold",
    "-o", str(bids_out),
]

res = subprocess.run(cmd, capture_output=True, text=True)

if not dicom_dir.exists():
    raise FileNotFoundError(f"DICOM folder not found: {dicom_dir}")

cmd = ["dcm2bids_helper", "-d", str(dicom_dir), "-o", str(helper_out)]
print("Running:", " ".join(cmd))

subprocess.run(cmd, check=True)

series_desc = set()
protocols = set()

json_files = sorted(help_2.glob("*.json"))
print("JSON sidecars found:", len(json_files))

for j in json_files:
    obj = json.loads(j.read_text(encoding="utf-8"))
    if "SeriesDescription" in obj:
        series_desc.add(obj["SeriesDescription"])
    if "ProtocolName" in obj:
        protocols.add(obj["ProtocolName"])

print("\nUnique SeriesDescription values:")
for s in sorted(series_desc):
    print("  -", s)

print("\nUnique ProtocolName values:")
for p in sorted(protocols):
    print("  -", p)

T1_NAME = "3D T1"      # example
FLAIR_NAME = "3D_Brain_VIEW_FLAIR_SAG"   # example

config = {
    "descriptions": [
        {
            "datatype": "anat",
            "suffix": "T1w",
            "criteria": {
                "SeriesDescription": T1_NAME
            }
        },
        {
            "datatype": "anat",
            "suffix": "FLAIR",
            "criteria": {
                "SeriesDescription": FLAIR_NAME
            }
        }
    ]
}

# 🔹 Where to save config

config_path.parent.mkdir(parents=True, exist_ok=True)

with open(config_path, "w") as f:
    json.dump(config, f, indent=4)

print("Config saved at:", config_path)


bids_out.mkdir(parents=True, exist_ok=True)

subprocess.run([
    "dcm2bids",
    "-d", dicom_dir,
    "-p", sub,
    "-s", "BL",
    "-c", str(config_path),
    "-o", str(bids_out)
], check=True)

print("✅ BIDS created at:", bids_out)